In [140]:
import pandas as pd

movies = pd.read_csv("movies.csv")
actors = pd.read_csv("actors.csv")
crew = pd.read_csv("crew.csv")
genres = pd.read_csv("genres.csv")
themes = pd.read_csv("themes.csv")



In [141]:
print(movies.shape)
print(movies.head())

print(actors.shape)
print(crew.shape)
print(genres.shape)

(941597, 7)
        id                               name    date  \
0  1000001                             Barbie  2023.0   
1  1000002                           Parasite  2019.0   
2  1000003  Everything Everywhere All at Once  2022.0   
3  1000004                         Fight Club  1999.0   
4  1000005                         La La Land  2016.0   

                                            tagline  \
0                  She's everything. He's just Ken.   
1                       Act like you own the place.   
2  The universe is so much bigger than you realize.   
3                           Mischief. Mayhem. Soap.   
4                    Here's to the fools who dream.   

                                         description  minute  rating  
0  Barbie and Ken are having the time of their li...   114.0    3.86  
1  All unemployed, Ki-taek's family takes peculia...   133.0    4.56  
2  An aging Chinese immigrant is swept up in an i...   140.0    4.30  
3  A ticking-time-bomb insomni

In [142]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941597 entries, 0 to 941596
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id           941597 non-null  int64  
 1   name         941587 non-null  object 
 2   date         849684 non-null  float64
 3   tagline      139387 non-null  object 
 4   description  780785 non-null  object 
 5   minute       760027 non-null  float64
 6   rating       90999 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 50.3+ MB


In [143]:
movies = movies[movies["rating"].notna()]
movies = movies[movies["description"].notna()]

In [144]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90403 entries, 0 to 166579
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           90403 non-null  int64  
 1   name         90403 non-null  object 
 2   date         90403 non-null  float64
 3   tagline      38412 non-null  object 
 4   description  90403 non-null  object 
 5   minute       89983 non-null  float64
 6   rating       90403 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 5.5+ MB


In [145]:
movies = movies[movies["rating"] >= 3.0]

In [146]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69024 entries, 0 to 164079
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           69024 non-null  int64  
 1   name         69024 non-null  object 
 2   date         69024 non-null  float64
 3   tagline      25481 non-null  object 
 4   description  69024 non-null  object 
 5   minute       68676 non-null  float64
 6   rating       69024 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 4.2+ MB


In [147]:
# rename some cols
movies = movies.rename(columns={
    "name": "title",
    "date": "year"
})
# Convert into year
movies["year"] = pd.to_numeric(
    movies["year"],
    errors="coerce"
)

In [148]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


movies["description"] = movies["description"].apply(clean_text)
movies["tagline"] = movies["tagline"].apply(clean_text)
movies["title"] = movies["title"].apply(clean_text)

In [149]:
genre_data = (
    genres.groupby("id")["genre"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

genre_data.head()

,id,genre
0,1000001,Comedy Adventure
1,1000002,Comedy Thriller Drama
2,1000003,Science Fiction Adventure Comedy Action
3,1000004,Drama
4,1000005,Drama Comedy Music Romance


In [150]:
movies = movies.merge(
    genre_data,
    on="id",
    how="left"
)

In [151]:
theme_data = (
    themes.groupby("id")["theme"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

movies = movies.merge(
    theme_data,
    on="id",
    how="left"
)

In [152]:
actor_data = (
    actors.groupby("id")["name"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

actor_data.rename(
    columns={"name": "actors"},
    inplace=True
)

movies = movies.merge(
    actor_data,
    on="id",
    how="left"
)

In [153]:
movies.head()

,id,title,year,tagline,description,minute,rating,genre,theme,actors
0,1000001,barbie,2023.0,she s everything he s just ken,barbie and ken are having the time of their li...,114.0,3.86,Comedy Adventure,Humanity and the world around us Crude humor a...,Margot Robbie Ryan Gosling America Ferrera Ari...
1,1000002,parasite,2019.0,act like you own the place,all unemployed ki taek s family takes peculiar...,133.0,4.56,Comedy Thriller Drama,Humanity and the world around us Intense viole...,Song Kang-ho Lee Sun-kyun Cho Yeo-jeong Choi W...
2,1000003,everything everywhere all at once,2022.0,the universe is so much bigger than you realize,an aging chinese immigrant is swept up in an i...,140.0,4.30,Science Fiction Adventure Comedy Action,Humanity and the world around us Moving relati...,Michelle Yeoh Ke Huy Quan Stephanie Hsu James ...
3,1000004,fight club,1999.0,mischief mayhem soap,a ticking time bomb insomniac and a slippery s...,139.0,4.27,Drama,Intense violence and sexual transgression Huma...,Edward Norton Brad Pitt Helena Bonham Carter M...
4,1000005,la la land,2016.0,here s to the fools who dream,mia an aspiring actress serves lattes to movie...,129.0,4.09,Drama Comedy Music Romance,Song and dance Humanity and the world around u...,Ryan Gosling Emma Stone John Legend Rosemarie ...


In [154]:
directors = crew[
    crew["role"].str.lower().str.contains("director", na=False)
]

In [155]:
director_data = (
    directors.groupby("id")["name"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

director_data.rename(
    columns={"name": "directors"},
    inplace=True
)

movies = movies.merge(
    director_data,
    on="id",
    how="left"
)

In [156]:
movies["content"] = (
    movies["genre"].fillna("") + " " +
    movies["genre"].fillna("") + " " +

    movies["theme"].fillna("") + " " +

    movies["directors"].fillna("") + " " +
    movies["directors"].fillna("") + " " +

    movies["actors"].fillna("") + " " +

    movies["description"].fillna("")
)

In [157]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    dtype=np.float32
)

In [158]:
tfidf_matrix = tfidf.fit_transform(movies["content"])

In [159]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 4278584 stored elements and shape (69024, 30000)>
  Coords	Values
  (0, 5090)	0.03334975242614746
  (0, 517)	0.05272641405463219
  (0, 12188)	0.03390153497457504
  (0, 29259)	0.07091299444437027
  (0, 5791)	0.0334688238799572
  (0, 12205)	0.03177947551012039
  (0, 23827)	0.06617274880409241
  (0, 18750)	0.028888095170259476
  (0, 22322)	0.050948332995176315
  (0, 25738)	0.025908665731549263
  (0, 8057)	0.02897978574037552
  (0, 3856)	0.03358534723520279
  (0, 8740)	0.027903985232114792
  (0, 25794)	0.03836256265640259
  (0, 26122)	0.036404598504304886
  (0, 26891)	0.035883579403162
  (0, 21607)	0.037449803203344345
  (0, 28402)	0.038929615169763565
  (0, 15943)	0.021216794848442078
  (0, 6279)	0.029890256002545357
  (0, 21764)	0.034683503210544586
  (0, 8102)	0.038268767297267914
  (0, 22327)	0.03034447319805622
  (0, 1143)	0.0381498821079731
  (0, 13808)	0.03422781452536583
  :	:
  (69023, 21910)	0.1329103410243988
  (69023

In [160]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    n_neighbors=51,
    metric="cosine",
    algorithm="brute"
)

model.fit(tfidf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=51)

In [161]:
distances, indices = model.kneighbors(
    tfidf_matrix[100],
    n_neighbors=11
)
print(distances,indices)

[[0.         0.62933457 0.64963406 0.6523124  0.655815   0.67159843
  0.67358404 0.67479694 0.6790329  0.68101406 0.68425924]] [[  100 20773   616 15907  2053 43345  3931 68301  2705  9720 53792]]


In [162]:
similarity = 1 - distances

In [163]:
import joblib

joblib.dump(tfidf, "tfidf_vectorizer.pkl", compress = 3)
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl", compress = 3)
joblib.dump(model, "model.pkl")
movies.to_pickle("movies.pkl")

print("All model files saved successfully!")

All model files saved successfully!


# Building the recommendation function

In [164]:
# creating a movie lookup
movie_indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

In [165]:
def recommend_movies(title, n=10):

    movie_index = movies[
        movies["title"].str.lower() == title.lower()
    ].index

    if len(movie_index) == 0:
        return "Movie not found"

    movie_index = movie_index[0]

    distances, indices = model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=n + 1
    )

    recommendations = []

    for distance, index in zip(
        distances[0][1:],
        indices[0][1:]
    ):

        # Cosine similarity
        similarity = 1 - distance

        # Movie's normalized rating
        rating_norm = movies.iloc[index]["rating"] / 5

        # Final recommendation score
        final_score = (
            0.75 * similarity +
            0.25 * rating_norm
        )

        recommendations.append({
            "Movie": movies.iloc[index]["title"],
            "Similarity": round(similarity, 3),
            "Rating": movies.iloc[index]["rating"],
            "Final Score": round(final_score, 3)
        })

    recommendations = pd.DataFrame(recommendations)

    # Sort by final score
    recommendations = recommendations.sort_values(
        by="Final Score",
        ascending=False
    )
    return recommendations

In [166]:
recommend_movies("inception")

,Movie,Similarity,Rating,Final Score
1,interstellar,0.353,4.35,0.483
0,tenet,0.353,3.40,0.435
2,the dark knight rises,0.315,3.74,0.423
3,dunkirk,0.284,3.77,0.401
4,ending the knight,0.276,3.76,0.395
6,oppenheimer,0.242,4.23,0.393
9,the dark knight,0.224,4.47,0.392
5,dreams cinema of the subconscious,0.263,3.54,0.374
8,kingdom of the planet of the apes,0.226,3.40,0.339
7,tron legacy,0.229,3.15,0.330


In [167]:
posters = pd.read_csv("posters.csv")

print(posters.columns)
print(posters.head())

Index(['id', 'link'], dtype='object')
        id                                               link
0  1000001  https://a.ltrbxd.com/resized/film-poster/2/7/7...
1  1000002  https://a.ltrbxd.com/resized/film-poster/4/2/6...
2  1000003  https://a.ltrbxd.com/resized/film-poster/4/7/4...
3  1000004  https://a.ltrbxd.com/resized/film-poster/5/1/5...
4  1000005  https://a.ltrbxd.com/resized/film-poster/2/4/0...


In [168]:
posters = posters.rename(columns={"link": "poster_url"})